# ClearVQA Analysis with Qwen2.5-VL

This notebook demonstrates a minimal Python/Jupyter workflow for:
- Loading and exploring the ClearVQA dataset
- Running visual question answering with a Qwen2.5-VL model


## 1) Environment setup

Run this once in the notebook if dependencies are missing:
```bash
pip install -q datasets transformers accelerate pillow qwen-vl-utils
```


In [ ]:
from collections import Counter
from datasets import load_dataset
from PIL import Image

DATASET_ID = "lmms-lab/ClearVQA"
SPLIT = "train"

dataset = load_dataset(DATASET_ID, split=SPLIT)
print(f"Loaded {len(dataset)} samples from {DATASET_ID} ({SPLIT})")
dataset[0]


In [ ]:
# Basic schema/statistics
print("Columns:", dataset.column_names)

# If the dataset includes question type/category metadata, summarize it.
possible_type_cols = ["question_type", "category", "type"]
type_col = next((c for c in possible_type_cols if c in dataset.column_names), None)

if type_col:
    counts = Counter(dataset[type_col])
    print(f"Top values in '{type_col}':")
    for label, count in counts.most_common(10):
        print(f"  {label}: {count}")
else:
    print("No question type/category column found.")


In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None
)
if not torch.cuda.is_available():
    model = model.to(device)

processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"Loaded model: {MODEL_ID} on {device}")


In [ ]:
def run_vqa(image: Image.Image, question: str, max_new_tokens: int = 64) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    if not torch.cuda.is_available():
        inputs = {k: v.to(device) for k, v in inputs.items()}

    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return output_text[0]


In [ ]:
# Run inference on one sample (adapt key names if your dataset variant differs).
sample = dataset[0]

image_key = "image" if "image" in sample else next((k for k in sample if "image" in k.lower()), None)
question_key = "question" if "question" in sample else next((k for k in sample if "question" in k.lower()), None)
answer_key = "answer" if "answer" in sample else next((k for k in sample if "answer" in k.lower()), None)

if image_key is None or question_key is None:
    raise KeyError(f"Could not detect image/question keys in sample. Keys: {list(sample.keys())}")

image = sample[image_key]
question = sample[question_key]
prediction = run_vqa(image, question)

print("Question:", question)
print("Prediction:", prediction)
if answer_key:
    print("Ground truth:", sample[answer_key])
